# M3 — Transformer Fine-tuning

Fine-tune a pretrained Transformer for SafePost AI's 3-class social-media moderation task.

**Classes:** `HATE_SPEECH`, `OFFENSIVE_LANGUAGE`, `NEUTRAL`

### Goals
- Use the same train/validation/test split as the BiLSTM baseline.
- Fine-tune a pretrained Hugging Face Transformer.
- Capture validation and test metrics.
- Generate a confusion matrix.
- Export the fine-tuned model and tokenizer to `models/`.
- Capture conclusions for `docs/experiments/M3-transformer.md`.

> This notebook is intentionally self-contained. Reusable training code is deferred until the productionization milestones.


## 1. Experiment Configuration

Keep configuration in one place so the experiment is reproducible.


In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

SEED = 42

# Adjust these paths/column names to match the BiLSTM experiment artifacts.
TRAIN_PATH = Path("../data/processed/train.csv")
VALIDATION_PATH = Path("../data/processed/val.csv")
TEST_PATH = Path("../data/processed/test.csv")

TEXT_COLUMN = "tweet"
LABEL_COLUMN = "class"

# Expected integer label mapping: 0, 1, 2.
LABEL_NAMES = [
    "hate_speech",
    "offensive_language",
    "neutral",
]

MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = Path("../models/m3-transformer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = 128
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Model:", MODEL_NAME)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


Model: distilbert-base-uncased
Device: cpu


## 2. Dependencies

From the repository root, add the experiment dependencies with `uv`:

```bash
uv add transformers datasets accelerate scikit-learn matplotlib seaborn
```

Run the notebook using the project environment, for example:

```bash
uv run jupyter notebook
```


In [2]:
import transformers
import datasets
import sklearn

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("scikit-learn:", sklearn.__version__)


c:\Users\Sumit Das\experiments\app-development\safepost-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 5.16.1
datasets: 5.0.1
scikit-learn: 1.9.0


## 3. Load the Same Split Used by the BiLSTM Baseline

Do **not** create a new random split. The Transformer must use the exact same split so M4 comparison is meaningful.


In [3]:
train_df = pd.read_csv(TRAIN_PATH)
validation_df = pd.read_csv(VALIDATION_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

display(train_df.head())


Train: (17347, 2)
Validation: (3718, 2)
Test: (3718, 2)


,class,tweet
0,1,LMFAOOOOOO GAY AS FUCK RT @PubesOnFleeK: &#128...
1,1,RT @OGTREEZ: Steve found out you can't call Ta...
2,2,@SBPart01 @melodybrooke11 fuzzy dice might be ...
3,1,@Midnight_Snacka now fuck that stfu shit how a...
4,2,Shut up birds it's bed time.


In [4]:
for name, frame in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    print(f"\n{name.upper()}")
    display(frame[LABEL_COLUMN].value_counts(dropna=False))



TRAIN


class
1    13432
2     2914
0     1001
Name: count, dtype: int64


VALIDATION


class
1    2879
2     624
0     215
Name: count, dtype: int64


TEST


class
1    2879
2     625
0     214
Name: count, dtype: int64

## 4. Dataset Validation


In [5]:
expected_labels = set(range(len(LABEL_NAMES)))

for name, frame in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    assert frame[TEXT_COLUMN].notna().all(), f"Missing text in {name}"
    assert frame[LABEL_COLUMN].notna().all(), f"Missing labels in {name}"
    unexpected = set(frame[LABEL_COLUMN].unique()) - expected_labels
    assert not unexpected, f"Unexpected labels in {name}: {unexpected}"

print("Dataset validation passed.")


Dataset validation passed.


## 5. Load Pretrained Tokenizer and Model

The initial candidate is **DistilBERT**. If another model is selected, update `MODEL_NAME` and record the decision in an ADR.


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

id2label = {i: name.upper() for i, name in enumerate(LABEL_NAMES)}
label2id = {name.upper(): i for i, name in enumerate(LABEL_NAMES)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id,
)

print("Loaded:", MODEL_NAME)


c:\Users\Sumit Das\experiments\app-development\safepost-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sumit Das\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2506.79it/s]
[transfo

Loaded: distilbert-base-uncased


## 6. Convert to Hugging Face Dataset


In [7]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_pandas(
        train_df[[TEXT_COLUMN, LABEL_COLUMN]], preserve_index=False
    ),
    "validation": Dataset.from_pandas(
        validation_df[[TEXT_COLUMN, LABEL_COLUMN]], preserve_index=False
    ),
    "test": Dataset.from_pandas(
        test_df[[TEXT_COLUMN, LABEL_COLUMN]], preserve_index=False
    ),
})

dataset


DatasetDict({
    train: Dataset({
        features: ['tweet', 'class'],
        num_rows: 17347
    })
    validation: Dataset({
        features: ['tweet', 'class'],
        num_rows: 3718
    })
    test: Dataset({
        features: ['tweet', 'class'],
        num_rows: 3718
    })
})

## 7. Tokenization

Use the pretrained Hugging Face tokenizer. Keep preprocessing inline for this experiment.


In [8]:
def tokenize_batch(batch):
    return tokenizer(
        batch[TEXT_COLUMN],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_batch, batched=True)
tokenized_dataset = tokenized_dataset.rename_column(LABEL_COLUMN, "labels")
tokenized_dataset = tokenized_dataset.remove_columns([TEXT_COLUMN])
tokenized_dataset.set_format("torch")

tokenized_dataset


Map: 100%|██████████| 3718/3718 [00:00<00:00, 10707.70 examples/s]


DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 17347
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3718
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3718
    })
})

## 8. Evaluation Metrics

Macro F1 is important because the moderation classes may not be balanced.


In [9]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
    }


## 9. Fine-tune with Hugging Face Trainer


In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "training"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()


c:\Users\Sumit Das\experiments\app-development\safepost-ai\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 10. Validation Evaluation


In [ ]:
validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
pd.DataFrame([validation_metrics])


## 11. Test Evaluation


In [ ]:
test_metrics = trainer.evaluate(tokenized_dataset["test"])
pd.DataFrame([test_metrics])


## 12. Detailed Test Metrics


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_output = trainer.predict(tokenized_dataset["test"])
test_predictions = np.argmax(test_output.predictions, axis=-1)
test_labels = test_output.label_ids

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=LABEL_NAMES,
        digits=4,
        zero_division=0,
    )
)


## 13. Confusion Matrix


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(
    test_labels,
    test_predictions,
    labels=list(range(len(LABEL_NAMES))),
)

plt.figure(figsize=(7, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("M3 Transformer — Test Confusion Matrix")
plt.tight_layout()
plt.show()


## 14. Export Model + Tokenizer

Artifacts remain gitignored and can later be packaged for SageMaker.


In [ ]:
EXPORT_DIR = OUTPUT_DIR / "final"

trainer.save_model(str(EXPORT_DIR))
tokenizer.save_pretrained(str(EXPORT_DIR))

print("Saved to:", EXPORT_DIR.resolve())


## 15. Qualitative Prediction Check

This is a quick sanity check, not a replacement for test-set metrics.


In [ ]:
from torch.nn.functional import softmax

def predict_text(text):
    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    with torch.no_grad():
        output = model(**encoded)

    probabilities = softmax(output.logits, dim=-1)[0].cpu().numpy()
    predicted_id = int(np.argmax(probabilities))

    return {
        "text": text,
        "label": LABEL_NAMES[predicted_id],
        "confidence": float(probabilities[predicted_id]),
        "probabilities": {
            LABEL_NAMES[i]: float(probabilities[i])
            for i in range(len(LABEL_NAMES))
        },
    }

examples = [
    "This is a normal discussion about today's news.",
    "I strongly dislike this person's opinion.",
    "Example text to inspect manually.",
]

for example in examples:
    print(predict_text(example))


## 16. Experiment Summary

Fill this section after completing the experiment.

### Configuration

- Model:
- Tokenizer:
- Max sequence length:
- Epochs:
- Learning rate:
- Batch size:
- Seed:

### Results

| Metric | Validation | Test |
|---|---:|---:|
| Accuracy | | |
| Macro Precision | | |
| Macro Recall | | |
| Macro F1 | | |

### Per-class observations

- Hate Speech:
- Offensive Language:
- Neutral:

### Comparison with BiLSTM

- Accuracy:
- Macro F1:
- Training time:
- Inference considerations:
- Model size:

### Key observations

1.
2.
3.

### Decision

- [ ] Transformer is better than BiLSTM
- [ ] BiLSTM remains preferable
- [ ] More investigation required

### Next Action

Proceed to M4 model comparison and selection.


## 17. M3 Definition-of-Done Checklist

- [ ] Notebook executes top-to-bottom using `uv run`.
- [ ] Same split as BiLSTM baseline confirmed.
- [ ] Model/version recorded.
- [ ] Hyperparameters recorded.
- [ ] Validation metrics captured.
- [ ] Test metrics captured.
- [ ] Confusion matrix captured.
- [ ] Fine-tuned model exported.
- [ ] Tokenizer exported.
- [ ] `docs/experiments/M3-transformer.md` created.
- [ ] Roadmap updated.
